# LoRA
IndoBERT (`indobenchmark/indobert-base-p1`), 3 seeds, 100% of training data.

Configuration from the sensitivity check: **r=4, lr=1e-4**, targeting the
query/value projection matrices in self-attention. `lora_alpha=8` follows
the alpha = 2r convention used for the r=8 pilot run -- adjust if your
sensitivity check settled on a different alpha.

In [ ]:
!pip install -q -U transformers datasets peft accelerate scikit-learn

import transformers, datasets, peft, torch
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("peft        :", peft.__version__)
print("torch       :", torch.__version__)
print("CUDA avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

# Record these version numbers after this first run and pin them
# (e.g. !pip install transformers==X.Y.Z) in the other method notebooks --
# a version drift between your FFT run and your LoRA run would make the
# VRAM/time comparison unreliable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/thesis/code')  # <- folder where thesis_utils.py lives
from thesis_utils import *

print("Data dir      :", DATA_DIR)
print("Results file  :", RESULTS_PATH)
print("Predictions   :", PREDICTIONS_PATH)
print("Seeds         :", SEEDS)

In [ ]:
# Optional, run once (not per-notebook): confirms train/test don't share
# exact-duplicate titles. Safe to skip on later notebooks once you've
# checked this the first time.
_ = check_train_test_leakage()

### Model definition
Low-rank adapters injected into query/value projections; base weights frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

LORA_R = 4
LORA_ALPHA = 8       # alpha = 2r, matches the r=8/alpha=16 convention from the pilot
LORA_DROPOUT = 0.1

def build_lora_model():
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
    )
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["query", "value"],   # BERT self-attention proj layer names
        bias="none",
    )
    model = get_peft_model(base, lora_config)
    model.print_trainable_parameters()
    return model

### Run all seeds
LR = 1e-4, per the sensitivity check.

In [ ]:
METHOD_NAME = "LoRA"
LEARNING_RATE = 1e-4

results_this_method = []
for seed in SEEDS:
    row = run_experiment(
        method_name=METHOD_NAME,
        seed=seed,
        model_init_fn=build_lora_model,
        learning_rate=LEARNING_RATE,
    )
    if row is not None:
        results_this_method.append(row)

### Summary for this method

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_PATH)
df[df["method"] == METHOD_NAME]